# Introduction to Web Scraping (Colab)

In [1]:
!pip install -q requests beautifulsoup4
!mkdir -p web-scraping-intro/src web-scraping-intro/data

### `src/__init__.py`

In [2]:
%%writefile web-scraping-intro/src/__init__.py
# web-scraping-intro/src/__init__.py
# This file indicates that the 'src' directory is a Python package.


Writing web-scraping-intro/src/__init__.py


### `src/scraper.py`

In [3]:
%%writefile web-scraping-intro/src/scraper.py
# web-scraping-intro/src/scraper.py
"""Simple web scraper for the "Automate the Boring Stuff with Python" homepage."""

import re
import time
from urllib.parse import urljoin, urlparse
from urllib.robotparser import RobotFileParser

import requests
from bs4 import BeautifulSoup

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
)

# Table-of-contents items: "Introduction", "Chapter 1 - ...", "Appendix A - ..."
TOC_PATTERN = re.compile(r"^(Introduction|Chapter\s+\d+|Appendix\s+[A-Z])\b", re.IGNORECASE)


class SimpleWebScraper:
    """A class to encapsulate basic web scraping functionality."""

    def __init__(self, target_url, delay=1.0):
        self.target_url = target_url
        self.delay = delay  # seconds to wait before each request (rate limiting)
        self.headers = {"User-Agent": USER_AGENT}

    def is_allowed_by_robots(self):
        """Check robots.txt to see whether we are allowed to scrape the target URL."""
        parts = urlparse(self.target_url)
        robots_url = f"{parts.scheme}://{parts.netloc}/robots.txt"
        try:
            response = requests.get(robots_url, headers=self.headers, timeout=10)
            if response.status_code != 200:
                return True  # no robots.txt -> no rules to follow
            parser = RobotFileParser()
            parser.parse(response.text.splitlines())
            return parser.can_fetch(USER_AGENT, self.target_url)
        except requests.exceptions.RequestException:
            return True  # could not read robots.txt -> continue carefully

    def _get_html_content(self):
        """Download the HTML content from the target URL with basic error handling."""
        print(f"กำลังดาวน์โหลดเนื้อหาจาก: {self.target_url}")
        time.sleep(self.delay)  # be polite to the server
        try:
            response = requests.get(self.target_url, headers=self.headers, timeout=10)
            response.raise_for_status()  # raise HTTPError for 4xx / 5xx
            print("ดาวน์โหลดเนื้อหาสำเร็จ")
            return response.text
        except requests.exceptions.HTTPError as e:
            print(f"เกิดข้อผิดพลาด HTTP: {e} - Status Code: {e.response.status_code}")
        except requests.exceptions.ConnectionError:
            print(f"เชื่อมต่อไม่ได้: {self.target_url}")
        except requests.exceptions.Timeout:
            print("หมดเวลาในการเชื่อมต่อ (timeout)")
        except requests.exceptions.RequestException as e:
            print(f"เกิดข้อผิดพลาดที่ไม่คาดคิด: {e}")
        return None

    def parse_titles(self, html_content):
        """Parse HTML and return the book title and chapter titles/links.

        Kept separate from downloading so it can be tested with saved HTML.
        """
        soup = BeautifulSoup(html_content, "html.parser")

        # Book title: the first <h1> on the page, or the <title> tag as a fallback
        title_tag = soup.find("h1") or soup.find("title")
        book_title = title_tag.get_text(strip=True) if title_tag else "ไม่พบชื่อหนังสือ"

        # Chapter titles: every <a> whose text starts with Introduction / Chapter N / Appendix X
        chapter_titles = []
        chapter_links = []
        for link in soup.find_all("a"):
            text = " ".join(link.get_text().split())  # clean up extra spaces/newlines
            text = text.replace("–", "-")             # use a normal hyphen everywhere
            if TOC_PATTERN.match(text) and text not in chapter_titles:
                chapter_titles.append(text)
                chapter_links.append(urljoin(self.target_url, link.get("href", "")))

        return {
            "book_title": book_title,
            "chapter_titles": chapter_titles,
            "chapter_links": chapter_links,
        }

    def scrape_main_titles(self):
        """Download the page and extract the book title and chapter titles."""
        if not self.is_allowed_by_robots():
            print("robots.txt ไม่อนุญาตให้ scrape หน้านี้ หยุดการทำงาน")
            return None

        html_content = self._get_html_content()
        if not html_content:
            return None
        return self.parse_titles(html_content)


Writing web-scraping-intro/src/scraper.py


### `src/data_processor.py`

In [4]:
%%writefile web-scraping-intro/src/data_processor.py
# web-scraping-intro/src/data_processor.py
"""Helpers for saving scraped data."""

import json
import os


def save_to_json(data, file_path):
    """Save a dictionary to a JSON file, creating the folder if needed."""
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print(f"บันทึกข้อมูลลงไฟล์: {file_path}")


Writing web-scraping-intro/src/data_processor.py


### `main.py`

In [5]:
%%writefile web-scraping-intro/main.py
# web-scraping-intro/main.py
"""Entry point for the simple web scraping application.

Usage:
    python main.py          # scrape and print results
    python main.py --open   # also open the target page in your default browser
"""

import os
import sys
import webbrowser

# Add the 'src' directory to the Python path
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
sys.path.append(os.path.join(BASE_DIR, "src"))

from scraper import SimpleWebScraper
from data_processor import save_to_json

TARGET_URL = "https://automatetheboringstuff.com/3e/"
OUTPUT_FILE = os.path.join(BASE_DIR, "data", "scraped_output.json")


def main():
    """Run the scraper, print the results and save them to JSON."""
    if "--open" in sys.argv:
        webbrowser.open(TARGET_URL)  # webbrowser module: open URL in default browser

    scraper = SimpleWebScraper(TARGET_URL)
    scraped_data = scraper.scrape_main_titles()

    if not scraped_data:
        print("ไม่สามารถ Scrape ข้อมูลได้")
        return

    print("\n--- ข้อมูลที่ Scrape ได้ ---")
    print(f"ชื่อหนังสือ: {scraped_data['book_title']}")
    print("\nชื่อบทต่างๆ:")
    if scraped_data["chapter_titles"]:
        for i, title in enumerate(scraped_data["chapter_titles"], start=1):
            print(f"{i}. {title}")
    else:
        print("ไม่พบชื่อบท")
    print("--------------------")

    save_to_json(scraped_data, OUTPUT_FILE)


if __name__ == "__main__":
    main()


Writing web-scraping-intro/main.py


### Run the scraper

In [6]:
!cd web-scraping-intro && python main.py

กำลังดาวน์โหลดเนื้อหาจาก: https://automatetheboringstuff.com/3e/
ดาวน์โหลดเนื้อหาสำเร็จ

--- ข้อมูลที่ Scrape ได้ ---
ชื่อหนังสือ: 3rd Edition

ชื่อบทต่างๆ:
1. Introduction
2. Chapter 1 - Python Basics
3. Chapter 2 - if-else and Flow Control
4. Chapter 3 - Loops
5. Chapter 4 - Functions
6. Chapter 5 - Debugging
7. Chapter 6 - Lists
8. Chapter 7 - Dictionaries and Structuring Data
9. Chapter 8 - Strings and Text Editing
10. Chapter 9 - Text Pattern Matching with Regular Expressions
11. Chapter 10 - Reading and Writing Files
12. Chapter 11 - Organizing Files
13. Chapter 12 - Designing and Deploying Command Line Programs
14. Chapter 13 - Web Scraping
15. Chapter 14 - Excel Spreadsheets
16. Chapter 15 - Google Sheets
17. Chapter 16 - SQLite Databases
18. Chapter 17 - PDF and Word Documents
19. Chapter 18 - CSV, JSON, and XML Files
20. Chapter 19 - Keeping Time, Scheduling Tasks, and Launching Programs
21. Chapter 20 - Sending Email, Texts, and Push Notifications
22. Chapter 21 - Making Gra

### View the saved JSON

In [7]:
!cat web-scraping-intro/data/scraped_output.json

{
  "book_title": "3rd Edition",
  "chapter_titles": [
    "Introduction",
    "Chapter 1 - Python Basics",
    "Chapter 2 - if-else and Flow Control",
    "Chapter 3 - Loops",
    "Chapter 4 - Functions",
    "Chapter 5 - Debugging",
    "Chapter 6 - Lists",
    "Chapter 7 - Dictionaries and Structuring Data",
    "Chapter 8 - Strings and Text Editing",
    "Chapter 9 - Text Pattern Matching with Regular Expressions",
    "Chapter 10 - Reading and Writing Files",
    "Chapter 11 - Organizing Files",
    "Chapter 12 - Designing and Deploying Command Line Programs",
    "Chapter 13 - Web Scraping",
    "Chapter 14 - Excel Spreadsheets",
    "Chapter 15 - Google Sheets",
    "Chapter 16 - SQLite Databases",
    "Chapter 17 - PDF and Word Documents",
    "Chapter 18 - CSV, JSON, and XML Files",
    "Chapter 19 - Keeping Time, Scheduling Tasks, and Launching Programs",
    "Chapter 20 - Sending Email, Texts, and Push Notifications",
    "Chapter 21 - Making Graphs and Manipulating Images",